# MA3632 — Workshop 1: Python Foundations and Data Acquisition

This workshop accompanies Lecture 1. Part A recaps the Python and pandas tools used
throughout the module. Part B covers data acquisition from several source types,
culminating in the `acquisition_report` function you will reuse in later workshops.
Work through both parts in order; the exercises at the end are for consolidation.

---

## Part A — Python and pandas recap

If you are already comfortable with lists, dictionaries, NumPy arrays, and basic
DataFrame operations, skim through and complete Exercise A to check your footing.
If anything here is unfamiliar, work through the comments carefully before moving on.

### A1. Core Python structures

In [ ]:
scores = [72, 85, 61, 90, 78, 55, 88]

mean_score = sum(scores) / len(scores)
print(f"Mean score: {mean_score:.2f}")
print(f"Max: {max(scores)}, Min: {min(scores)}")

# Filter with a list comprehension
high_scores = [s for s in scores if s >= 80]
print(f"High scorers (>=80): {high_scores}")

# Dictionary
student_info = {
    "name": "Alice",
    "cohort": 2024,
    "modules": ["Data Mining", "Statistics", "Python Programming"]
}
print(f"\nStudent: {student_info['name']}, Cohort: {student_info['cohort']}")
print(f"Modules: {', '.join(student_info['modules'])}")

### A2. NumPy

In [ ]:
import numpy as np

data = np.array([2.1, 3.4, np.nan, 1.7, 4.0, np.nan, 3.1])

print("Array:", data)
print(f"Mean (ignoring NaN): {np.nanmean(data):.3f}")
print(f"Std  (ignoring NaN): {np.nanstd(data):.3f}")
print(f"NaN positions: {np.where(np.isnan(data))[0]}")

# Standardise — no explicit loop needed
scaled = (data - np.nanmean(data)) / np.nanstd(data)
print(f"\nStandardised: {np.round(scaled, 2)}")

### A3. pandas

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "student_id": range(1, 8),
    "score":      [72, 85, 61, 90, 78, 55, 88],
    "attended":   [True, True, False, True, True, False, True],
    "grade":      ["B", "A", "C", "A", "B", "D", "A"]
})

print(df)
print("\nData types:\n", df.dtypes)

In [ ]:
# Select, filter, aggregate, derive

print("Scores only:", df["score"].values)

print("\nStudents who attended and scored >= 80:")
print(df[(df["attended"]) & (df["score"] >= 80)])

print("\nMean score by attendance:")
print(df.groupby("attended")["score"].mean().round(2))

df["passed"] = df["score"] >= 60
print(f"\nPass rate: {df['passed'].mean()*100:.1f}%")

**Exercise A.** Using the DataFrame `df` above:

Calculate the median score for each grade category. Then add a column `score_band`
that labels each student as `"low"` (score < 65), `"mid"` (65-84), or `"high"` (>= 85),
and report how many students fall in each band.

---

## Part B — Data acquisition

Data acquisition is the first stage of the pipeline. In practice data arrives from many
sources — databases, flat files, APIs, sensors, or synthetic generation — and rarely in
a clean state. This section covers three of the most common acquisition routes.

### B1. Built-in datasets

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

try:
    housing = fetch_california_housing(as_frame=True)
except Exception:
    # Offline fallback: synthetic stand-in with the same column structure
    import numpy as np, types
    np.random.seed(0)
    n = 200
    data = {
        "MedInc":      np.random.uniform(1, 10, n),
        "HouseAge":    np.random.uniform(1, 52, n),
        "AveRooms":    np.random.uniform(2, 10, n),
        "AveBedrms":   np.random.uniform(1, 3,  n),
        "Population":  np.random.randint(100, 3000, n).astype(float),
        "AveOccup":    np.random.uniform(1, 5,  n),
        "Latitude":    np.random.uniform(32, 42, n),
        "Longitude":   np.random.uniform(-124, -114, n),
        "MedHouseVal": np.random.uniform(0.5, 5, n),
    }
    feature_descriptions = {
        "MedInc":     "Median income in block group (tens of thousands of dollars)",
        "HouseAge":   "Median house age in block group (years)",
        "AveRooms":   "Average number of rooms per household",
        "AveBedrms":  "Average number of bedrooms per household",
        "Population": "Block group population",
        "AveOccup":   "Average number of household members",
        "Latitude":   "Block group latitude",
        "Longitude":  "Block group longitude",
    }
    housing = types.SimpleNamespace(
        frame=pd.DataFrame(data),
        feature_names=list(feature_descriptions.keys()),
        feature_descriptions=feature_descriptions,
    )
    print("Note: using synthetic fallback (network unavailable).")

df_housing = housing.frame
print("Shape:", df_housing.shape)
print("\nFeature descriptions:")
if hasattr(housing, "feature_descriptions"):
    for name, desc in housing.feature_descriptions.items():
        print(f"  {name:<20} {desc}")
else:
    for name, desc in zip(housing.feature_names, housing.DESCR.split("\n")[12:20]):
        print(f"  {name:<20} {desc.strip()}")

In [ ]:
print(df_housing.head(3))
print("\nDescriptive statistics:")
print(df_housing.describe().round(2))
print("\nMissing values:")
print(df_housing.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_housing["MedHouseVal"], bins=50, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Median house value ($100k)")
axes[0].set_ylabel("Count")
axes[0].set_title("California Housing — target distribution")

corr = df_housing.corr(numeric_only=True)
im = axes[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr)))
axes[1].set_yticks(range(len(corr)))
axes[1].set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
axes[1].set_yticklabels(corr.columns, fontsize=8)
axes[1].set_title("Correlation matrix")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

top = corr["MedHouseVal"].drop("MedHouseVal").abs().idxmax()
print(f"Feature most correlated with house value: {top}")
print("(With the full dataset this should be MedInc — median income.)")

The target variable is right-skewed. Consider what this implies for a linear regression
model fitted on the raw values, and what transformation might address it.

### B2. Synthetic data generation

Synthetic data is useful when real data is unavailable — due to privacy constraints, say,
or when one wants to test a pipeline under conditions where the ground truth is known.
The two examples below correspond to the cases in Lecture 1.

In [ ]:
from sklearn.datasets import make_classification

X_cls, y_cls = make_classification(
    n_samples=500,
    n_features=6,
    n_informative=3,   # only 3 features carry signal
    n_redundant=1,     # 1 feature is a linear combination of the informative ones
    n_classes=2,
    weights=[0.7, 0.3],  # class imbalance
    random_state=42
)

df_cls = pd.DataFrame(X_cls, columns=[f"feature_{i}" for i in range(6)])
df_cls["label"] = y_cls

print("Shape:", df_cls.shape)
print("Class distribution:\n", df_cls["label"].value_counts())

In [ ]:
# Manual construction: known correlation structure and deliberate missing values
np.random.seed(0)
n = 300

age       = np.random.randint(18, 65, size=n)
income    = 20000 + age * 800 + np.random.normal(0, 5000, n)  # correlated with age
education = np.random.choice(["secondary", "undergraduate", "postgraduate"], n,
                              p=[0.3, 0.5, 0.2])

income_obs = income.astype(float)
income_obs[np.random.choice(n, size=20, replace=False)] = np.nan

df_synthetic = pd.DataFrame({"age": age, "income": income_obs, "education": education})

print(df_synthetic.head(8))
print(f"\nMissing income values: {df_synthetic['income'].isna().sum()} / {n}")
print(f"Income-age correlation: {df_synthetic[['age','income']].corr().iloc[0,1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df_synthetic["age"], df_synthetic["income"],
                alpha=0.4, s=20, color="darkorange")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Income")
axes[0].set_title("Synthetic dataset: age vs income")

edu_counts = df_synthetic["education"].value_counts()
axes[1].bar(edu_counts.index, edu_counts.values,
            color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_xlabel("Education level")
axes[1].set_ylabel("Count")
axes[1].set_title("Synthetic dataset: education")

plt.tight_layout()
plt.show()

**Exercise B.2.** The missing values in `income` above were introduced at random,
independently of any other variable. What is the missingness mechanism under Rubin's
taxonomy? How would your answer change if instead the missing values had been
concentrated among the youngest observations?

### B3. External files

The most common external formats in practice are CSV, TSV, and JSON.
This section demonstrates JSON acquisition using a small dataset constructed
and written to disk locally — no network connection required.
An optional extension shows the same workflow with a remote CSV.

In [ ]:
import json, pathlib, datetime

# Construct a small JSON dataset and write it to disk
records = [
    {"id": 1,  "age": 34, "income": 52000, "education": "undergraduate",  "employed": True},
    {"id": 2,  "age": 27, "income": None,   "education": "secondary",      "employed": False},
    {"id": 3,  "age": 45, "income": 81000, "education": "postgraduate",   "employed": True},
    {"id": 4,  "age": 52, "income": 67000, "education": "undergraduate",  "employed": True},
    {"id": 5,  "age": 23, "income": 21000, "education": "secondary",      "employed": False},
    {"id": 6,  "age": 38, "income": None,   "education": "postgraduate",   "employed": True},
    {"id": 7,  "age": 31, "income": 44000, "education": "undergraduate",  "employed": True},
    {"id": 8,  "age": 29, "income": 38000, "education": "secondary",      "employed": False},
    {"id": 9,  "age": 41, "income": 59000, "education": "postgraduate",   "employed": True},
    {"id": 10, "age": 36, "income": 47000, "education": "undergraduate",  "employed": True},
]

path = pathlib.Path("survey_data.json")
with open(path, "w") as f:
    json.dump(records, f, indent=2)

print(f"Written {len(records)} records to {path}.")
print(f"File size: {path.stat().st_size} bytes")

In [ ]:
with open("survey_data.json") as f:
    loaded = json.load(f)

df_survey = pd.DataFrame(loaded)
print("Shape:", df_survey.shape)
print()
print(df_survey)
print()
print("Dtypes:")
print(df_survey.dtypes)
print()
print("Note: income is float64 because JSON null -> Python None -> pandas NaN,")
print("which forces the column to floating-point.")

In [ ]:
def acquisition_report(df, name="dataset"):
    """Concise audit summary for a DataFrame.  Reused in all later workshops."""
    n, p = df.shape
    missing = df.isnull().sum()
    missing_pct = (missing / n * 100).round(1)

    print(f"=== {name} ===")
    print(f"  Rows: {n}   Columns: {p}")
    print(f"  Dtypes: {dict(df.dtypes.value_counts())}")
    print()
    print("  Missing values:")
    if missing.sum() == 0:
        print("    None")
    else:
        for col in missing[missing > 0].sort_values(ascending=False).index:
            print(f"    {col:<20} {missing[col]:>4} ({missing_pct[col]}%)")
    print()
    print("  Duplicate rows:", df.duplicated().sum())
    print()
    print("  Numeric summary:")
    print(df.describe().round(2).to_string())
    print()

acquisition_report(df_survey, "Survey")

In [ ]:
print("Categorical columns:")
cat_cols = df_survey.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    vc = df_survey[col].value_counts()
    print(f"  {col}: {dict(vc)}")

**Optional extension — URL acquisition (requires internet access).**

If you have a working connection, the cell below loads a CSV from a public URL
and applies `acquisition_report` to it.  If the URL is unreachable, skip this cell;
no later part of the workshop depends on it.

In [ ]:
# Optional — network required
try:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df_titanic = pd.read_csv(url)
    print(f"Loaded from URL. Shape: {df_titanic.shape}")
    acquisition_report(df_titanic, "Titanic (URL)")

    print("Categorical columns:")
    cat_cols_t = df_titanic.select_dtypes(include=["object", "string"]).columns
    for col in cat_cols_t:
        vc = df_titanic[col].value_counts()
        print(f"  {col}: {dict(list(vc.items())[:5])}{'...' if len(vc) > 5 else ''}")
except Exception as e:
    print(f"URL not reachable ({e}). Skipping — this cell is optional.")

**Exercise B.3.**  Two of the ten survey records have `income` missing.
Based on the dataset as constructed, what can you say about the missingness mechanism
under Rubin's taxonomy?  What additional information would you need to determine
whether it is MAR rather than MCAR?

### B4. Provenance

In [ ]:
import datetime

provenance = {
    "dataset":   "Employee survey (synthetic)",
    "source":    "Constructed in-session (Workshop 1, B3)",
    "acquired":  datetime.date.today().isoformat(),
    "n_records": len(df_survey),
    "licence":   "No licence required — synthetic data",
    "caveats":   ("2 missing values in income (mechanism unknown without further "
                  "context). Values constructed for pedagogical illustration only."),
}

print("Provenance record")
print("-" * 40)
for k, v in provenance.items():
    print(f"  {k:<12}: {v}")

print()
print("Note: if the optional Titanic URL cell was run, a second provenance record")
print("should be completed for that dataset (source URL, licence, known caveats).")

---

## Take-home exercises

**Exercise 1.** Apply `acquisition_report` to `df_housing`. What are the main data
quality observations?

**Exercise 2.** Construct a synthetic dataset with 200 rows, two correlated numerical
features, one ordinal feature (three levels), and 5% missing values in one column
introduced at random. Use NumPy directly rather than `make_classification`.

**Exercise 3.** Fill in a provenance record (using the template from B4) for the
dataset you created in Exercise 2. What do you write for `source` and `licence`?

**Exercise 4.** (Written, no code.) Give two real-world situations in which synthetic
data generation would be appropriate, and one situation in which it would be
insufficient. Justify each answer.

---